# Modeling and Simulation 5 - Advanced Monte Carlo Methods

## Variance Reduction and Quasi-Monte Carlo

In this lab we will explore advanced techniques to improve Monte Carlo simulations. We will:

1. Estimate the constant **e** using a Monte Carlo stopping-time method.
2. Apply **variance reduction** using antithetic variates.
3. Compare results with **quasi-Monte Carlo** (low-discrepancy sequences).

All code should be fully functional. You will implement the missing parts marked with `# TODO`.

### 1. Monte Carlo Estimation of e

Let $ X_1, X_2, \dots $ be i.i.d. $ \text{Uniform}(0,1) $. Define $ N $ as:

$$ N = \min\{k \in \mathbb{N} : X_1 + \cdots + X_k > 1\}. $$

**Theorem:** $ \mathbb{E}[N] = e $.

*Proof:* The event $ \{N > n\} $ occurs iff $ \sum_{i=1}^{n} X_i \leq 1 $. The probability equals the volume of the simplex 

$$ \left\{ (x_1,\dots,x_n) \in [0,1]^n : \sum_{i=1}^{n} x_i \leq 1 \right\}, $$

which is $ 1/n! $. Therefore:

$$ \mathbb{E}[N] = \sum_{n=0}^{\infty} \Pr(N > n) = \sum_{n=0}^{\infty} \frac{1}{n!} = e. $$

We estimate $ e $ by sampling $ N_1,\dots,N_M $ i.i.d. and computing $ \hat{e}_M = \frac{1}{M}\sum_{i=1}^{M} N_i $. The variance is $ \text{Var}(\hat{e}_M) = \text{Var}(N)/M $.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

### Task 1: Implement the basic estimator

Complete `sample_N()` and `estimate_e_mc(num_trials)`.

- `sample_N()`: draws uniforms until sum > 1, returns the count.
- `estimate_e_mc()`: runs many trials, returns the sample mean and the array of all $ N $ values.

After implementation, run with 10,000 trials and compute the standard error and a 95% confidence interval.

In [ ]:
def sample_N():
    """Return one realization of N: count of Uniform(0,1) draws until sum > 1."""
    count = 0
    total = 0.0
    while total <= 1.0:
        x = np.random.uniform(0, 1)
        total += x
        count += 1
    return count

def estimate_e_mc(num_trials):
    """Run num_trials independent samples of N.
    Returns: (mean, array_of_N)
    """
    N_values = np.empty(num_trials, dtype=int)
    for i in range(num_trials):
        N = sample_N()
        N_values[i] = N
    mean = np.mean(N_values)
    return mean, N_values

# Test
np.random.seed(42)
e_hat, N_array = estimate_e_mc(10000)
se_hat = np.std(N_array, ddof=1) / np.sqrt(len(N_array))
print(f"Estimate: {e_hat:.6f}")
print(f"Std error: {se_hat:.6f}")
print(f"True e: {np.e:.6f}")
ci_low = e_hat - 1.96 * se_hat
ci_high = e_hat + 1.96 * se_hat
print(f"95% CI: [{ci_low:.6f}, {ci_high:.6f}]")

#### Variance analysis

Run one large experiment to estimate $ \text{Var}(N) $. Then perform 500 replications of $ M=1000 $ trials each and compute the empirical variance of those 500 $ \hat{e}_M $ values. Compare with the theoretical $ \text{Var}(N)/M $.

In [ ]:
# Estimate Var(N) from a single large run
np.random.seed(123)
_, N_big = estimate_e_mc(20000)
var_N = np.var(N_big, ddof=1)
print(f"Estimated Var(N) = {var_N:.6f}")

# 500 replications of M=1000
num_rep = 500
M = 1000
estimates = np.empty(num_rep)
for r in range(num_rep):
    e_r, _ = estimate_e_mc(M)
    estimates[r] = e_r

var_estimates = np.var(estimates, ddof=1)
theoretical_var = var_N / M
print(f"\nEmpirical variance of \\hat{{e}}_M over {num_rep} replications: {var_estimates:.6f}")
print(f"Theoretical (Var(N)/M): {theoretical_var:.6f}")
print(f"Ratio (empirical/theoretical): {var_estimates/theoretical_var:.3f}")

# Plot distribution of estimates
plt.figure(figsize=(8,4))
plt.hist(estimates, bins=20, edgecolor='black', alpha=0.7)
plt.axvline(np.e, color='red', linestyle='--', label='True e')
plt.xlabel('Estimate of e')
plt.ylabel('Frequency')
plt.title(f'Distribution of {num_rep} independent MC estimates (M=1000)')
plt.legend()
plt.grid(True)
plt.show()

## 2. Variance Reduction via Antithetic Variates

If $ X $ and $ X' $ are negatively correlated with $ \mathbb{E}[X] = \mathbb{E}[X'] $, then $ (X+X')/2 $ has lower variance than $ X $.

For uniforms, define $ X' = 1 - X $. Then $ \text{Cov}(X, 1-X) = -\text{Var}(X) < 0 $.

**Method:** For each trial, use the same uniforms $ U_{i,j} $ to generate two sub-trials:
- Trial A: use $ U_{i,j} $
- Trial B: use $ 1-U_{i,j} $
Both yield $ N_A $ and $ N_B $ with $ \mathbb{E}[N_A] = \mathbb{E}[N_B] = e $. The antithetic average is:

$$ N_{\text{anti}} = \frac{N_A + N_B}{2}. $$

Its variance:

$$ \text{Var}(N_{\text{anti}}) = \tfrac{1}{4}\big( \text{Var}(N_A) + \text{Var}(N_B) + 2\text{Cov}(N_A, N_B) \big). $$

Since $ \text{Cov}(N_A, N_B) < 0 $, we get variance reduction.

**Implementation strategy:**

1. Pre-generate a matrix $ U $ of shape $ (P, L) $ where $ P $ = number of pairs, $ L $ = max draws per pair (e.g., 50).
2. For each row $ i $:
   - Compute $ N_A $ by accumulating $ U_{i,0}, U_{i,1}, \dots $ until sum > 1.
   - Compute $ N_B $ by accumulating $ 1-U_{i,0}, 1-U_{i,1}, \dots $ until sum > 1.
   - Store $ (N_A + N_B)/2 $.

This yields $ P $ effective antithetic samples.

In [ ]:
def estimate_e_antithetic(num_pairs, max_draws=50):
    """Return (mean, array_of_averages) using antithetic variates.
    Pre-generates U (num_pairs x max_draws). For each row:
      - N_A from cumulative sum of U[i]
      - N_B from cumulative sum of 1-U[i]
      - store (N_A + N_B)/2
    """
    U = np.random.rand(num_pairs, max_draws)
    averages = np.empty(num_pairs)

    for i in range(num_pairs):
        # Compute N_A
        sum_A = 0.0
        N_A = 0
        for j in range(max_draws):
            sum_A += U[i, j]
            N_A += 1
            if sum_A > 1:
                break
        # Compute N_B
        sum_B = 0.0
        N_B = 0
        for j in range(max_draws):
            sum_B += (1 - U[i, j])
            N_B += 1
            if sum_B > 1:
                break
        averages[i] = (N_A + N_B) / 2.0

    return np.mean(averages), averages

# Test
np.random.seed(42)
e_anti, anti_vals = estimate_e_antithetic(5000)
print(f"Antithetic estimate: {e_anti:.6f}")
print(f"True e: {np.e:.6f}")

#### Variance reduction factor

Compare variances of naive (5000 samples) vs antithetic (5000 effective samples). Compute VRF = Var(naive)/Var(antithetic). Also compute the empirical covariance between $ N_A $ and $ N_B $ to confirm it is negative.

In [ ]:
# Get naive samples (using M=5000)
np.random.seed(42)
_, naive_samples = estimate_e_mc(5000)

# Get antithetic samples along with N_A, N_B for covariance
def estimate_e_antithetic_with_pairs(num_pairs, max_draws=50):
    U = np.random.rand(num_pairs, max_draws)
    averages = np.empty(num_pairs)
    N_A_array = np.empty(num_pairs, dtype=int)
    N_B_array = np.empty(num_pairs, dtype=int)

    for i in range(num_pairs):
        sum_A, N_A = 0.0, 0
        for j in range(max_draws):
            # TODO: implement inner loop            pass        sum_B, N_B = 0.0, 0
        for j in range(max_draws):
            # TODO: implement inner loop            pass        averages[i] = (N_A + N_B) / 2.0
        N_A_array[i] = N_A
        N_B_array[i] = N_B

    return np.mean(averages), averages, N_A_array, N_B_array

np.random.seed(42)
_, anti_samples, NA, NB = estimate_e_antithetic_with_pairs(5000)

var_naive = np.var(naive_samples, ddof=1)
var_anti = np.var(anti_samples, ddof=1)
cov_ANB = np.cov(NA, NB, ddof=1)[0,1]

print(f"Naive variance: {var_naive:.6f}")
print(f"Antithetic variance: {var_anti:.6f}")
print(f"Variance reduction factor: {var_naive/var_anti:.3f}")
print(f"Cov(N_A, N_B): {cov_ANB:.6f}")

# Plot distributions
plt.figure(figsize=(9,4))
plt.hist(naive_samples, bins=range(1,10), alpha=0.6, label='Naive', density=True, edgecolor='black')
plt.hist(anti_samples, bins=range(1,10), alpha=0.6, label='Antithetic', density=True, edgecolor='black')
plt.axvline(np.e, color='red', linestyle='--', label='True e')
plt.xlabel('N')
plt.ylabel('Density')
plt.title('Naive vs Antithetic distribution (5000 effective samples)')
plt.legend()
plt.grid(True)
plt.show()

## 3. Quasi-Monte Carlo

### Discrepancy

Monte Carlo's error scales as $ O(1/\sqrt{N}) $. Quasi-Monte Carlo (QMC) uses deterministic low-discrepancy sequences (LDS) with **star discrepancy**:

$$ D_N^* = \sup_{\mathbf{u} \in [0,1]^d} \left| \frac{1}{N}\sum_{i=1}^{N} \mathbf{1}_{[0,\mathbf{u})}(\mathbf{x}_i) - \prod_{j=1}^{d} u_j \right|. $$

LDS achieve $ D_N^* = O((\log N)^d / N) $.

### Common LDS
- **Van der Corput** (base 2): radical inverse in 1D.
- **Sobol'**: base-2, good to very high dimensions.
- **Halton**: different prime bases per dimension.

For our e-problem, we replace random uniforms with an LDS. We sequentially consume the sequence across trials.

**Note:** Theoretical $ \mathbb{E}[N]=e $ assumes i.i.d. uniforms; QMC may be slightly biased but often converges faster in RMSE.

### Task
- Implement or import generators for Sobol' and Halton (use `scipy.stats.qmc` if available, else Van der Corput).
- Compare three methods: Mersenne Twister (random), Sobol', Halton.
- Plot convergence of running estimate vs number of trials.

In [ ]:
try:
    from scipy.stats import qmc
    has_scipy = True
except ImportError:
    has_scipy = False
    print("scipy.stats.qmc not available; using Van der Corput as fallback")

def van_der_corput(n, base=2):
    """Generate n points of the Van der Corput sequence in base `base`."""
    seq = []
    for i in range(1, n+1):
        v = 0.0
        f = 1.0 / base
        j = i
        while j > 0:
            v += (j % base) * f
            j //= base
            f /= base
        seq.append(v)
    return seq

def generate_sobol(n):
    if has_scipy:
        sampler = qmc.Sobol(d=1, scramble=False)
        points = sampler.random(n)
        return points.flatten()
    else:
        return van_der_corput(n, base=2)

def generate_halton(n):
    if has_scipy:
        sampler = qmc.Halton(d=1, scramble=False)
        points = sampler.random(n)
        return points.flatten()
    else:
        # In 1D, Halton with base=2 is same as Van der Corput
        return van_der_corput(n, base=2)

# Quick test
print("First 5 Sobol:", generate_sobol(5))
print("First 5 Halton:", generate_halton(5))

### Running comparison

We'll allocate 500,000 uniforms for each method. The helper `run_from_uniforms()` consumes them sequentially and returns the running mean after each completed trial.

In [ ]:
def run_from_uniforms(uni_array):
    """Consume uniforms sequentially to compute running estimate of e.
    Returns: (running_means_array, number_of_trials_completed)"""
    N_vals = []
    idx = 0
    running_means = []
    while idx < len(uni_array):
        total = 0.0
        count = 0
        while idx < len(uni_array) and total <= 1.0:
            total += uni_array[idx]
            idx += 1
            count += 1
        if total > 1:
            N_vals.append(count)
            running_means.append(np.mean(N_vals))
        else:
            # Ran out of uniforms mid-trial; discard incomplete trial
            break
    return np.array(running_means), len(N_vals)

def compare_qmc(total_draws=500000):
    np.random.seed(42)
    u_mc = np.random.rand(total_draws)
    u_sobol = np.array(generate_sobol(total_draws))
    u_halton = np.array(generate_halton(total_draws))

    run_mc, n_mc = run_from_uniforms(u_mc)
    run_sobol, n_sobol = run_from_uniforms(u_sobol)
    run_halton, n_halton = run_from_uniforms(u_halton)

    print(f"Trials completed: MC={n_mc}, Sobol={n_sobol}, Halton={n_halton}")
    print(f"Final errors: |MC-e|={abs(run_mc[-1]-np.e):.6f}, |Sobol-e|={abs(run_sobol[-1]-np.e):.6f}, |Halton-e|={abs(run_halton[-1]-np.e):.6f}")

    plt.figure(figsize=(10,5))
    plt.plot(run_mc, label='MC (random)', alpha=0.8)
    plt.plot(run_sobol, label='Sobol QMC', alpha=0.8)
    plt.plot(run_halton, label='Halton QMC', alpha=0.8)
    plt.axhline(np.e, color='black', linestyle='--', label='True e')
    plt.xlabel('Trial number')
    plt.ylabel('Running estimate of e')
    plt.title(f'MC vs QMC for estimating e ({total_draws} total draws)')
    plt.legend()
    plt.grid(True)
    plt.show()

compare_qmc(500000)

#### Analysis

1. Which method achieved the lowest final absolute error? By what margin?
2. Did any method exhibit systematic bias (consistently above or below e)?
3. How does the smoothness (local fluctuations) of the convergence curves compare?
4. (Optional) What happens if you use `scramble=True` in the QMC samplers?

## Conclusion

- Implemented a base Monte Carlo estimator for $ e $ via uniform-sum stopping time.
- Achieved variance reduction with antithetic variates (factor typically ~1.5–2×).
- Compared pseudorandom sampling with Sobol' and Halton low-discrepancy sequences.

**Takeaways:**

- Variance reduction can substantially improve precision for a given sample size.
- Quasi-Monte Carlo often converges faster for smooth integrands, though error bounds are deterministic.
- The choice of technique depends on cost of sampling and dimensionality.